# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [ ]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
'''
import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI
'''

In [ ]:
# Initialize and constants
'''
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()
'''

In [1]:
# Imports for Ollama setup
# Use this instead of OpenAI
import os
import requests
import json
from typing import List
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Test Ollama connection
try:
    response = requests.get('http://localhost:11434/api/tags', timeout=5)
    if response.status_code == 200:
        models = response.json().get('models', [])
        print(f"Ollama is running! Available models: {[model['name'] for model in models]}")
    else:
        print("Ollama server responded but with an error")
except requests.exceptions.ConnectionError:
    print("Cannot connect to Ollama. Make sure it's running on localhost:11434")
except requests.exceptions.Timeout:
    print("Connection to Ollama timed out")

# Configure OpenAI client to use Ollama
MODEL = 'llama3.2'  # Change this to your preferred model
openai = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # Required but can be any string for local Ollama
)

Ollama is running! Available models: ['llama3.2:1b']


In [10]:
# Initialize and constants for Ollama
import requests
from openai import OpenAI

# Test Ollama connection and get available models
try:
    response = requests.get('http://localhost:11434/api/tags', timeout=5)
    if response.status_code == 200:
        models_data = response.json().get('models', [])
        if models_data:
            model_names = [model['name'] for model in models_data]
            print(f"✅ Ollama connection successful! Available models: {model_names}")
            MODEL = model_names[0]  # Use first available model
            print(f"Using model: {MODEL}")
        else:
            print("❌ Ollama is running but no models installed!")
            print("Install a model with: ollama pull llama3.2:1b")
            MODEL = 'llama3.2:1b'  # Default fallback
    else:
        print("❌ Ollama server error")
        MODEL = 'llama3.2:1b'
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to Ollama. Make sure it's running with: ollama serve")
    MODEL = 'llama3.2:1b'
except Exception as e:
    print(f"❌ Error connecting to Ollama: {e}")
    MODEL = 'llama3.2:1b'

# Initialize OpenAI client to work with Ollama
openai = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama'  # Required but can be any string for local Ollama
)

✅ Ollama connection successful! Available models: ['llama3.2:1b']
Using model: llama3.2:1b


In [11]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [26]:
#ed = Website("https://edwarddonner.com")
ed = Website("https://www.tribalfootball.com")
ed.links

['/',
 '/',
 '/topic/football-transfers-dc8f04f4-a484-49e2-9203-e1b33ee0400d',
 '/topic/soccer-10b95a82-c5d4-4ac1-b320-19d0730801ed',
 '/topic/premier-league-e119c953-fa7a-4f00-93a3-9769df116bdd',
 '/topic/champions-league-e2d762bd-dd46-4f78-9068-65fc2eabb56b',
 '/topic/laliga-5ad0285d-5801-4917-bbcc-507670cb38ec',
 '/topic/bundesliga-28c51e52-c56c-4479-af62-7adb03a46921',
 '/topic/serie-a-ad716509-12d4-4430-9e86-53f17fdf0411',
 '/topic/ligue-1-94828ed3-d1d7-46b0-8bba-40d74fa6ed33',
 '/topic/europa-league-b9aa0eab-a9f1-4082-8ca9-706d0f0ddb7c',
 '/article/soccer-premier-league-amorim-reveals-sesko-onana-could-start-against-arsenal-insists-hojlund-still-man-utd-player-bf564d8d-7205-429d-92a3-70a3911bf926',
 '/article/soccer-premier-league-mount-admits-man-utd-cannot-afford-early-slip-against-arsenal-9f88625f-d0f1-48cd-8bec-b7d2875fbe72',
 '/article/soccer-premier-league-ex-brighton-striker-duffus-takes-jamaican-citizenship-to-join-saint-etienne-d4e69700-e53a-4089-a0df-cbae22b0ad77',
 '/a

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [27]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [28]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}



In [30]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [31]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://www.tribalfootball.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
/
/
/topic/football-transfers-dc8f04f4-a484-49e2-9203-e1b33ee0400d
/topic/soccer-10b95a82-c5d4-4ac1-b320-19d0730801ed
/topic/premier-league-e119c953-fa7a-4f00-93a3-9769df116bdd
/topic/champions-league-e2d762bd-dd46-4f78-9068-65fc2eabb56b
/topic/laliga-5ad0285d-5801-4917-bbcc-507670cb38ec
/topic/bundesliga-28c51e52-c56c-4479-af62-7adb03a46921
/topic/serie-a-ad716509-12d4-4430-9e86-53f17fdf0411
/topic/ligue-1-94828ed3-d1d7-46b0-8bba-40d74fa6ed33
/topic/europa-league-b9aa0eab-a9f1-4082-8ca9-706d0f0ddb7c
/article/soccer-premier-league-amorim-reveals-sesko-onana-could-start-against-arsenal-insists-hojlund-still-man-utd-player-bf564d8d-7205-429d-92a3-70a3911bf926
/article/soccer-premier-league

In [32]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [ ]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..
tribalfootball = Website("https://www.tribalfootball.com")
tribalfootball.links

In [34]:
#get_links("https://tribalfootball.com")
get_links("https://github.com")

{'serverTime': '2023-05-22 14:28:32.801',
 'sessionId': '1234567890',
 'userAgent': 'TribalFootball Compter',
 'locale': 'de',
 'os': 'Windows 10 19H2',
 'ipAddress': '192.168.1.100',
 'connectionType': 'WebSockets/ TLS 1.3',
 'language': 'German',
 ' referringDomain': 'www.tribalfootball.com'}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [35]:
def get_all_details(url):
    result = "Landing page:\n"
    result += Website(url).get_contents()
    links = get_links(url)
    print("Found links:", links)
    for link in links["links"]:
        result += f"\n\n{link['type']}\n"
        result += Website(link["url"]).get_contents()
    return result

In [37]:
#print(get_all_details("https://huggingface.co"))
#print(get_all_details("https://httpbin.org/html"))
#print(get_all_details("https://tribalfootball.com"))
print(get_all_details("https://github.com"))

Found links: {'server': 'decentralized-dns:**80**', 'domain': 'tribalfootball.com', 'command': 'headless', 'version': '1.0.12)', 'url': 'http://trustedthirdpartyadapters.com/443/en-US/api/v2/users/78912345/get_all_users/'}


KeyError: 'links'

In [ ]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    user_prompt += get_all_details(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("TribalFootball", "https://tribalfootball.com")

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("TribalFootball", "https://tribalfootball.com")

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>